# Random IBS Data Generation Script

This juoyter notebook seeks to expand on the current volume of EINS dataset by randomly generating new data given an adjusted set of parameters

Importing the libraries

In [1]:
import datetime
import pandas as pd
import json
import itertools
import random
import string
import numpy as np
import joblib
import sys

sys.path.append('../../../Litigation Clustering/')

from litigation_clustering import label_litigation_main_cause, label_litigation_case_status, label_litigation_party_status

c:\Users\65973\Downloads\GitHub files go here\Capstone-Project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Runner Configuration

In [2]:
with open('../../ibs_variables.json') as f:
    data = json.load(f)

In [3]:
def randdate(start_date, end_date, update=True):
    if end_date == start_date:
        return end_date
    time_between_dates = end_date - start_date
    days_between_dates = time_between_dates.days
    random_number_of_days = random.randrange(days_between_dates)
    random_date = start_date + datetime.timedelta(days=random_number_of_days)
    if update:
        global randomDate
        randomDate = random_date
    return random_date

def randdates(num, start_date, end_date, update=True):
    random_dates = []
    for _ in range(num):
        random_date = randdate(start_date, end_date, update=False)
        random_dates.append(random_date)
    if update:
        global randomDates
        randomDates = random_dates
    return random_dates



# Most recent 2 years section
def random_Related_Search_Count_Generation(individual_id, purposes, year):
    individual_id_col = [individual_id for i in range(13)] * (len(purposes) + 3)
    # Generate the 'month' and 'year' column
    months = ['All', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    month_col = months * (len(purposes) + 3)
    year_col = [year] * len(months) * (len(purposes) + 1)
    year_col.extend([year-1] * len(months))
    year_col.extend([year-2] * len(months))

    # Generate the purpose column    
    purpose_col = ['All'] * 13 + list(itertools.chain(*[[ele] * 13 for ele in purposes])) + ['All'] * 26


    # Most recent 2 years section
    def random_row_generation():
        try:
            randomAll = random.randint(0, 5)
            num_of_splits = random.randint(0, randomAll)
            splits = []
            for i in range(num_of_splits - 1):
                split = random.randrange(1, randomAll - sum(splits))
                splits.append(split)
            splits.append(randomAll - sum(splits))

            # Create the data row
            row = [randomAll]
            monthsValue = [0] * (12 - len(splits)) + splits
            random.shuffle(monthsValue)
            row.extend(monthsValue)
            return row
        except ValueError:
            return [0] * 13

    randomRows = [random_row_generation() for i in range(len(purposes))]
    totalSum = [sum(row) for row in zip(*randomRows)]
    
    count_col = totalSum + list(itertools.chain(*randomRows)) + random_row_generation() + random_row_generation()

    # Creation of Dataframe
    dataframe = pd.DataFrame(data= {
        'individual_id': individual_id_col,
        'year': year_col, 
        'month': month_col, 
        'purpose': purpose_col, 
        'count': count_col
    }, columns=['individual_id', 'year', 'month', 'purpose', 'count'])

    return dataframe

In [4]:
# Number of random datasets to generate
dataCount = 10

# Intialise date variable
global randomDate
global randomDates
req_date = randdate(datetime.datetime(2023, 1, 1), datetime.datetime.today())
reg_date = randdate(datetime.datetime(2000, 1, 1), req_date)

# Imported fields
MAIN_CAUSE = list(pd.read_csv('../../../Dataset/litigation_data/litigation_main_cause.csv')['Main Cause'])
CASE_STATUS = list(pd.read_csv('../../../Dataset/litigation_data/litigation_case_status.csv')['Case Status'])
PARTY_STATUS = list(pd.read_csv('../../../Dataset/litigation_data/litigation_party_status.csv')['Party (Plaintiff/Defendant) Status'])
ENTITY_TYPE_INDICATOR = list(pd.read_csv('../../../ibs/uen mapping/UEN Mapping.csv')['Entity-Type Indicator'])
COY_STATUS = data['COY_STATUS']
POSITION = data['POSITION']
WRIT_SUM_COURT_TYPE = data['WRIT_SUM_COURT_TYPE']
FINANCIAL_REL_SEARCH_COUNT_VAL = data['FINANCIAL_REL_SEARCH_COUNT_VAL']
COMMERCIAL_REL_SEARCH_COUNT_VAL = data['COMMERCIAL_REL_SEARCH_COUNT_VAL']
LEGAL_REL_SEARCH_COUNT_VAL = data['LEGAL_REL_SEARCH_COUNT_VAL']
streetNameFirst = data['streetNameFirst']
streetNameLast = data['streetNameLast']
firstName = data['firstName']
middleName = data['middleName']
sector = data['sector']
companyType = data['companyType']
humanFirstName = data['humanFirstName']
humanMiddleName = data['humanMiddleName']
humanLastName = data['humanLastName']
accreditationAwards = data['accreditationAwards']

# Duplicated fields (Singular / Composite)
ENTITY_ID = f'{random.choice(string.ascii_uppercase)}{random.randint(100, 999)}XXXX{random.choice(string.ascii_uppercase)}'
ENTITY_NAME = f'{random.choice(firstName)} {random.choice(middleName)} {random.choice(sector)} {random.choice(companyType)}'

# Multiple Occurence fields (variable according to each report)
NUM_OF_CURR_BUSINESS_COUNT = random.randint(1, 5)
NUM_OF_PREV_BUSINESSES_COUNT = random.randint(1, 5)
NUM_OF_SHAREHOLDERS = random.randint(1, 5)
NUM_OF_OFFICERS_AUTH_REP = random.randint(1, 8)
WRIT_SUM_ORGS_CLAIM_COUNT = random.randint(1, 3)
WINDING_UP_COUNT = random.randint(1, 5)
OSOAOPOM_COUNT = random.randint(1, 3)
ACRRED_COUNT = random.randint(1, 5)
BANKRUPTCY_COUNT = random.randint(1, 5)
INDV_LIT_SEARCH_COUNT = random.randint(1, 5)
BUSINESS_LIT_SEARCH_COUNT = random.randint(1, 5) 

INDIVIDUAL_NAME = [f'{random.choice(humanFirstName)}', f'{random.choice(humanMiddleName)} {random.choice(humanLastName)}']

# Predefined Fields
CURRBUSINESSES = [f'{random.choice(firstName)} {random.choice(middleName)} {random.choice(sector)} {random.choice(companyType)}' for i in range(NUM_OF_CURR_BUSINESS_COUNT)]
PREVBUSINESSES = [f'{random.choice(firstName)} {random.choice(middleName)} {random.choice(sector)} {random.choice(companyType)}' for i in range(NUM_OF_PREV_BUSINESSES_COUNT)]


# Initialise clustering models
litigation_main_cause_clustering_model = joblib.load('../../../Litigation Clustering/litigation_main_cause_clustering.pkl')
litigation_case_status_clustering_model = joblib.load('../../../Litigation Clustering/litigation_case_status_clustering.pkl')
litigation_party_status_clustering_model = joblib.load('../../../Litigation Clustering/litigation_party_status_clustering.pkl')

# Complex Field Function
def nameOfPrivateTrusteesAppointed():
    random_name = f'{random.choice(humanFirstName)}', f'{random.choice(humanMiddleName)} {random.choice(humanLastName)}'
    random_address = f'{random.choice(streetNameFirst)} {random.choice(streetNameLast)} #{random.randint(1, 30)}-{random.randint(1, 50)} SINGAPORE {random.randint(100000, 999999)}'
    return ' '.join(list(random_name)) + ' ' + random_address


random_params = {
    "Request Criteria": {
        'request_date':                             req_date.strftime('%Y/%m/%d'),
        'requestor_name':                           "SUBSCRIBER",
        'requested_individual_name':                ' '.join(INDIVIDUAL_NAME),
        'requested_id_no':                          ENTITY_ID,
        'request_id':                               "IBS SAMPLE",
    },
    "Search Record": {
        'individual_name':                          '\n'.join([f'{index + 1}) {" ".join(fullName)}' for index, fullName in enumerate(itertools.permutations(INDIVIDUAL_NAME, 2) )]),
        'individual_id':                            ENTITY_ID,
        'available_address':                        f'{random.choice(streetNameFirst)} {random.choice(streetNameLast)} #{random.randint(1, 30)}- {random.randint(1, 50)} SINGAPORE {random.randint(100000, 999999)}',
    },
    "Current Business": {
        'business_name':                            [f'{random.choice(firstName)} {random.choice(middleName)} {random.choice(sector)} {random.choice(companyType)}' for _ in range(NUM_OF_CURR_BUSINESS_COUNT)],
        'registration_no':                          [random.choice([
                                                        f'{"".join(random.choices(string.digits, k=8))}{random.choice(string.ascii_uppercase)}', 
                                                        f'{reg_date.year}{"".join(random.choices(string.digits, k=5))}{random.choice(string.ascii_uppercase)}', 
                                                        f'T{reg_date.year % 100}{random.choice(ENTITY_TYPE_INDICATOR)}{"".join(random.choices(string.digits, k=4))}{random.choice(string.ascii_uppercase)}', 
                                                    ]) for _ in range(NUM_OF_CURR_BUSINESS_COUNT)],
        'status':                                   [random.choice(COY_STATUS) for _ in range(NUM_OF_CURR_BUSINESS_COUNT)],
        'position':                                 ['\n'.join(random.sample(POSITION, random.randint(1, len(POSITION)))) for _ in range(NUM_OF_CURR_BUSINESS_COUNT)],
        'appointment_date':                         [reg_date.strftime('%Y/%m/%d') for _ in range(NUM_OF_CURR_BUSINESS_COUNT)],
        'current_directors_todate_count':           [random.randint(0, 10) for i in range(NUM_OF_CURR_BUSINESS_COUNT)],
        'current_secretaries_todate_count':         [random.randint(0, 10) for i in range(NUM_OF_CURR_BUSINESS_COUNT)],
        'current_alternate_directors_todate_count': [random.randint(0, 10) for i in range(NUM_OF_CURR_BUSINESS_COUNT)],
        'active_charges_todate_count':              [random.randint(0, 10) for i in range(NUM_OF_CURR_BUSINESS_COUNT)],
        'current_shareholders_todate_count':        [random.randint(0, 10) for i in range(NUM_OF_CURR_BUSINESS_COUNT)],
        'subsidiary_todate_count':                  [random.randint(0, 10) for i in range(NUM_OF_CURR_BUSINESS_COUNT)],
    },
    "Previous Business": {
        'business_name':                            PREVBUSINESSES,
        'registration_no':                          [random.choice([
                                                        f'{"".join(random.choices(string.digits, k=8))}{random.choice(string.ascii_uppercase)}', 
                                                        f'{reg_date.year}{"".join(random.choices(string.digits, k=5))}{random.choice(string.ascii_uppercase)}', 
                                                        f'T{reg_date.year % 100}{random.choice(ENTITY_TYPE_INDICATOR)}{"".join(random.choices(string.digits, k=4))}{random.choice(string.ascii_uppercase)}', 
                                                    ]) for i in range(NUM_OF_PREV_BUSINESSES_COUNT)],
        'status':                                   [random.choice(COY_STATUS) for _ in range(NUM_OF_PREV_BUSINESSES_COUNT)],
        'position':                                 [random.choice(POSITION) for _ in range(NUM_OF_PREV_BUSINESSES_COUNT)],
        'cessation_date':                           [randdate(datetime.datetime(1900, 1, 1), reg_date).strftime('%Y/%m/%d') for i in range(NUM_OF_PREV_BUSINESSES_COUNT)],
    },
    "Current Business Lit Trace": {
        'business_name':                            list(itertools.chain(*[[biz] * 5 for biz in CURRBUSINESSES])),
        'year':                                     list(itertools.chain(*[[f'{req_date.year}'] * 5 for _ in CURRBUSINESSES])),
        'enquiries':                                list(itertools.chain(*[['Litigation Trace: District Court\n($60,000 - $250,000)', 'Litigation Trace: High Court\n(> $250,000)', 'Litigation Trace: Magistrate Court\n(< $60,000)', 'OS/OA/OP/OM Trace', 'Winding Up Trace'] for biz in CURRBUSINESSES])),
        'current_year_after_9_Jan':                 [random.randint(0, 1) for _ in range(NUM_OF_CURR_BUSINESS_COUNT * 5)],
        'previous_year_before_9_Jan':               [random.randint(0, 1) for _ in range(NUM_OF_CURR_BUSINESS_COUNT * 5)],
    },
    "Business Lit Search": {
        'sn':                                       [i+1 for i in range(BUSINESS_LIT_SEARCH_COUNT)],
        'court_type':                               [random.choice(WRIT_SUM_COURT_TYPE) for _ in range(BUSINESS_LIT_SEARCH_COUNT)],
        'case_no':                                  [random.randint(1000, 9999) for _ in range(BUSINESS_LIT_SEARCH_COUNT)],
        'year':                                     [min(randomDates[i], randomDates[-i - 1]).year for i in range(len(randdates(2 * BUSINESS_LIT_SEARCH_COUNT, reg_date, req_date)) // 2)],
        'filing_date':                              [min(randomDates[i], randomDates[-i - 1]).strftime('%Y/%m/%d') for i in range(BUSINESS_LIT_SEARCH_COUNT)],
        'plaintiff':                                [f'{random.choice(firstName)} {random.choice(middleName)} {random.choice(sector)} {random.choice(companyType)}' for _ in range(BUSINESS_LIT_SEARCH_COUNT)],
        'case_status':                              [random.choice(CASE_STATUS) for i in range(BUSINESS_LIT_SEARCH_COUNT)],
        'case_status_date':                         [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(BUSINESS_LIT_SEARCH_COUNT)],
        'case_status_remarks':                      [np.nan for i in range(BUSINESS_LIT_SEARCH_COUNT)],
        'plaintiff_status':                         [random.choice(PARTY_STATUS) for i in range(BUSINESS_LIT_SEARCH_COUNT)],
        'plaintiff_status_date':                    [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(BUSINESS_LIT_SEARCH_COUNT)],
        'defendant_status':                         [random.choice(PARTY_STATUS) for i in range(BUSINESS_LIT_SEARCH_COUNT)],
        'defendant_status_date':                    [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(BUSINESS_LIT_SEARCH_COUNT)],
        'amount':                                   [f'SGD {random.randint(10000, 1000000)}' for i in range(BUSINESS_LIT_SEARCH_COUNT)],
        'last_update':                              [max(randomDates[i], randomDates[-i - 1]).strftime('%Y/%m/%d') for i in range(BUSINESS_LIT_SEARCH_COUNT)],
        'main_cause':                               [random.choice(MAIN_CAUSE) for i in range(BUSINESS_LIT_SEARCH_COUNT)],
    },
    "Bankruptcy Records": {
        'case_no':                                  [random.randint(100, 9999) for _ in range(BANKRUPTCY_COUNT)],
        'year':                                     [min(randomDates[i], randomDates[-i - 1]).year for i in range(len(randdates(2 * BANKRUPTCY_COUNT, reg_date, req_date)) // 2)],
        'filing_date':                              [min(randomDates[i], randomDates[-i - 1]).strftime('%Y/%m/%d') for i in range(BANKRUPTCY_COUNT)],
        'plaintiff':                                [f'{random.choice(firstName)} {random.choice(middleName)} {random.choice(sector)} {random.choice(companyType)}' for _ in range(BANKRUPTCY_COUNT)],
        'case_status':                              [random.choice(CASE_STATUS) for _ in range(BANKRUPTCY_COUNT)],
        'case_status_date':                         [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(BANKRUPTCY_COUNT)],
        'case_status_remarks':                      [np.nan for _ in range(BANKRUPTCY_COUNT)],
        'plaintiff_status':                         [random.choice(PARTY_STATUS) for _ in range(BANKRUPTCY_COUNT)],
        'plaintiff_status_date':                    [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(BANKRUPTCY_COUNT)],
        'defendant_status':                         [random.choice(PARTY_STATUS) for _ in range(BANKRUPTCY_COUNT)],
        'defendant_status_date':                    [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(BANKRUPTCY_COUNT)],
        'amount':                                   [f'SGD {random.randint(10000, 1000000)}' for _ in range(BANKRUPTCY_COUNT)],
        'last_update':                              [max(randomDates[i], randomDates[-i - 1]).strftime('%Y/%m/%d') for i in range(BANKRUPTCY_COUNT)],
        'name_of_private_trustees':                 [nameOfPrivateTrusteesAppointed() for _ in range(BANKRUPTCY_COUNT)],
        'individual_id':                            [ENTITY_ID for i in range(BANKRUPTCY_COUNT)],
    },
    "Individual Lit Search": {
        'sn':                                       [i+1 for i in range(INDV_LIT_SEARCH_COUNT)],
        'type':                                     [random.choice(WRIT_SUM_COURT_TYPE) for _ in range(INDV_LIT_SEARCH_COUNT)],
        'case_no':                                  [random.randint(1000, 9999) for _ in range(INDV_LIT_SEARCH_COUNT)],
        'year':                                     [min(randomDates[i], randomDates[-i - 1]).year for i in range(len(randdates(2 * INDV_LIT_SEARCH_COUNT, reg_date, req_date)) // 2)],
        'filing_date':                              [min(randomDates[i], randomDates[-i - 1]).strftime('%Y/%m/%d') for i in range(INDV_LIT_SEARCH_COUNT)],
        'plaintiff':                                [f'{random.choice(humanFirstName)} {random.choice(humanMiddleName)} {random.choice(humanLastName)}' for _ in range(INDV_LIT_SEARCH_COUNT)],
        'case_status':                              [random.choice(CASE_STATUS) for _ in range(INDV_LIT_SEARCH_COUNT)],
        'case_status_date':                         [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(INDV_LIT_SEARCH_COUNT)],
        'case_status_remarks':                      [np.nan for _ in range(INDV_LIT_SEARCH_COUNT)],
        'plaintiff_status':                         [random.choice(PARTY_STATUS) for _ in range(INDV_LIT_SEARCH_COUNT)],
        'plaintiff_status_date':                    [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(INDV_LIT_SEARCH_COUNT)],
        'defendant_status':                         [random.choice(PARTY_STATUS) for _ in range(INDV_LIT_SEARCH_COUNT)],
        'defendant_status_date':                    [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(INDV_LIT_SEARCH_COUNT)],
        'amount':                                   [f'SGD {random.randint(10000, 1000000)}' for _ in range(INDV_LIT_SEARCH_COUNT)],
        'last_update':                              [max(randomDates[i], randomDates[-i - 1]).strftime('%Y/%m/%d') for i in range(INDV_LIT_SEARCH_COUNT)],
        'main_cause':                               [random.choice(MAIN_CAUSE) for _ in range(INDV_LIT_SEARCH_COUNT)],
    },
    "Accreditation": {
        'business':                                 [f'{random.choice(firstName)} {random.choice(middleName)} {random.choice(sector)}' for _ in range(ACRRED_COUNT)],
        'award_name':                               [f'{random.choice(accreditationAwards)}' for i in range(ACRRED_COUNT)],
        'award_ranking':                            [f'{random.randint(2000, 2020)} Ranked {random.randint(1, 1000)}' for _ in range(ACRRED_COUNT)],
    },
    "Financial Related Search Count":{
        # 'individual_id':                             ENTITY_ID,
        'year':                                      f'{req_date.year-1}',
        'month':                                     None,
        'purpose':                                   list(random.sample(FINANCIAL_REL_SEARCH_COUNT_VAL, random.randint(1, len(FINANCIAL_REL_SEARCH_COUNT_VAL)))),
        'count':                                     None,
    },
    "Commercial Related Search Count":{
        # 'individual_id':                             ENTITY_ID,
        'year':                                      f'{req_date.year-1}',
        'month':                                     None,
        'purpose':                                   list(random.sample(COMMERCIAL_REL_SEARCH_COUNT_VAL, random.randint(1, len(COMMERCIAL_REL_SEARCH_COUNT_VAL)))),
        'count':                                     None,
    },
    "Legal Related Search Count":{
        # 'individual_id':                             ENTITY_ID,
        'year':                                      f'{req_date.year-1}',
        'month':                                     None,
        'purpose':                                   list(random.sample(LEGAL_REL_SEARCH_COUNT_VAL, random.randint(1, len(LEGAL_REL_SEARCH_COUNT_VAL)))),
        'count':                                     None,
    },
}

c:\Users\65973\Downloads\GitHub files go here\Capstone-Project\.venv\lib\site-packages\sklearn\base.py:318: UserWarning: Trying to unpickle estimator KMeans from version 1.2.1 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [5]:
df_dict = pd.read_excel('../../../ibs/Output/ibs_sample_2.xlsx', sheet_name=None, usecols=lambda col: col != 'Unnamed: 0')

Table 1: Request Criteria

In [6]:
print('Given Input')
df_dict['Request Criteria']

Given Input


,Date of Request,Name of Requestor,Requested Individual Name,Requested ID Number,File Reference Number
0,09/01/20XX,SUBSCRIBER,TAN AH KOW,S123XXXXE,IBS SAMPLE


In [7]:
print('Generated Output')
random_req_criteria = pd.DataFrame(
    data=[list(random_params['Request Criteria'].values())], columns=random_params['Request Criteria'].keys()
)
random_req_criteria

Generated Output


,request_date,requestor_name,requested_individual_name,requested_id_no,request_id
0,2023/02/20,SUBSCRIBER,MAY SOUZ LEQUAN,R607XXXXK,IBS SAMPLE


Table 2: Search Record

In [8]:
print('Given Input')
df_dict['Search Record']

Given Input


,Individual Name (As per Experian database),ID Number,Available Address (As per Experian database)
0,1) TAN AH KOW \n2) MICHAEL TAN AH KOW \n3) TAN...,S123XXXXE,123 ANG MO KIO #06-239 \nSINGAPORE 123456


In [9]:
print('Generated Output')
random_search_rec = pd.DataFrame(
    data=[list(random_params['Search Record'].values())], columns=random_params['Search Record'].keys()
)
random_search_rec

Generated Output


,individual_name,individual_id,available_address
0,1) MAY SOUZ LEQUAN\n2) SOUZ LEQUAN MAY,R607XXXXK,JURONG AVENUE #12- 5 SINGAPORE 457732


Table 3: Current Business

In [10]:
print('Given Input')
df_dict['Current Business']

Given Input


,No.,Name,Registration Number,Status,Position,Appointment Date,Position Name,To-date Counts
0,1,ABC HOLDINGS PTE LTD,1990XXXXH,LIVE,DIRECTOR \nSHAREHOLDER,23/01/19XX,Current Directors,2
1,1,ABC HOLDINGS PTE LTD,1990XXXXH,LIVE,DIRECTOR \nSHAREHOLDER,23/01/19XX,Current Secretaries,0
2,1,ABC HOLDINGS PTE LTD,1990XXXXH,LIVE,DIRECTOR \nSHAREHOLDER,23/01/19XX,Current Alternate Directors,0
3,1,ABC HOLDINGS PTE LTD,1990XXXXH,LIVE,DIRECTOR \nSHAREHOLDER,23/01/19XX,Active Charges,2
4,1,ABC HOLDINGS PTE LTD,1990XXXXH,LIVE,DIRECTOR \nSHAREHOLDER,23/01/19XX,Current Shareholders / Owners,2
5,1,ABC HOLDINGS PTE LTD,1990XXXXH,LIVE,DIRECTOR \nSHAREHOLDER,23/01/19XX,Subsidiary/Associates,0
6,2,WWW PTE LTD,20000XXXXW,LIVE,DIRECTOR \nSHAREHOLDER,01/02/20XX,Current Directors,3
7,2,WWW PTE LTD,20000XXXXW,LIVE,DIRECTOR \nSHAREHOLDER,01/02/20XX,Current Secretaries,1
8,2,WWW PTE LTD,20000XXXXW,LIVE,DIRECTOR \nSHAREHOLDER,01/02/20XX,Current Alternate Directors,0
9,2,WWW PTE LTD,20000XXXXW,LIVE,DIRECTOR \nSHAREHOLDER,01/02/20XX,Active Charges,1


In [11]:
print('Generated Output')
random_current_business = pd.DataFrame(
    data=random_params['Current Business'], columns=random_params['Current Business'].keys()
)
random_current_business

Generated Output


,business_name,registration_no,status,position,appointment_date,current_directors_todate_count,current_secretaries_todate_count,current_alternate_directors_todate_count,active_charges_todate_count,current_shareholders_todate_count,subsidiary_todate_count
0,DUSK RIVER COAST CORP,T4CP8339S,CONVERTED TO LLP,SHAREHOLDER\nDIRECTOR,2004/10/08,1,1,4,10,5,4
1,DUSK RIVER REGIONAL CORP,06871501G,REMOVED,OWNER,2004/10/08,9,3,7,2,1,7
2,DAWN WATER INTERNATIONAL CORP,23252104G,UNDER JUDICIAL MANAGEMENT,OWNER\nSHAREHOLDER\nDIRECTOR,2004/10/08,7,0,3,5,4,9
3,TWILIGHT PLASTIC CENTRAL ENTERPRISE,200419380O,CONVERTED TO LLP,OWNER\nDIRECTOR,2004/10/08,4,9,4,1,5,9
4,MOONLIGHT PLASTIC REGIONAL ENTERPRISE,T4CC0413I,SUSPENDED,DIRECTOR\nOWNER\nSHAREHOLDER,2004/10/08,5,0,7,7,3,2


Table 4: Previous Busines

In [12]:
print('Given Input')
df_dict['Previous Business']

Given Input


,No.,Name,Registration Number,Status,Position,Cessation Date
0,1,ABCDE PTE LTD,19960XXXXC,LIVE,DIRECTOR,09/11/19XX
1,2,NETWORK DENKI,5021XXXXW,TERMINATION,OWNER,21/10/20XX
2,3,PIONEER INSTRUMENTS,5038XXXXC,CANCELLATION,OWNER,09/11/19XX


In [13]:
print('Generated Output')
randon_previous_business = pd.DataFrame(
    data=random_params['Previous Business'], columns=random_params['Previous Business'].keys()
)
randon_previous_business

Generated Output


,business_name,registration_no,status,position,cessation_date
0,SUNSHINE GREENYARN STATE PTE LIMITED,42963119Y,CANCELLED,OWNER,1962/07/31
1,MIDNIGHT METAL REGIONAL PTE LIMITED,200426997P,CANCELLED,SHAREHOLDER,1916/11/04
2,SUNSET WOOD REGIONAL CORP,200437218I,NA,SHAREHOLDER,1980/10/19
3,SUNSET WATER INTERNATIONAL PTE LIMITED,200434484H,LIVE,OWNER,1950/01/20


Table 5: Current Business Lit Trace

In [14]:
print('Given Input')
df_dict['Current Business Lit Trace']

Given Input


,Business,Enquiries,Current Year On/After 9 Jan 20XX,Previous Year Before 9 Jan 20XX
0,ABC HOLDINGS PTE LTD,"Litigation Trace: Magistrate Court \n(< $60,000)",0,1
1,ABC HOLDINGS PTE LTD,"Litigation Trace: District Court \n($60,000 -...",1,0
2,ABC HOLDINGS PTE LTD,"Litigation Trace: High Court \n(> $250,000)",0,0
3,ABC HOLDINGS PTE LTD,Winding Up Trace,0,0
4,ABC HOLDINGS PTE LTD,OS/OA/OP/OM Trace,0,0
5,WWW PTE LTD,"Litigation Trace: Magistrate Court \n(< $60,000)",0,1
6,WWW PTE LTD,"Litigation Trace: District Court \n($60,000 -...",1,0
7,WWW PTE LTD,"Litigation Trace: High Court \n(> $250,000)",0,0
8,WWW PTE LTD,Winding Up Trace,0,0
9,WWW PTE LTD,OS/OA/OP/OM Trace,0,0


In [15]:
print('Generated Output')
randomm_curr_business_lit_trace = pd.DataFrame(
    data=random_params['Current Business Lit Trace'], columns=random_params['Current Business Lit Trace'].keys()
)
randomm_curr_business_lit_trace

Generated Output


,business_name,year,enquiries,current_year_after_9_Jan,previous_year_before_9_Jan
0,MIDNIGHT AIR STATE PTE LIMITED,2023,"Litigation Trace: District Court\n($60,000 - $...",0,0
1,MIDNIGHT AIR STATE PTE LIMITED,2023,"Litigation Trace: High Court\n(> $250,000)",0,1
2,MIDNIGHT AIR STATE PTE LIMITED,2023,"Litigation Trace: Magistrate Court\n(< $60,000)",1,0
3,MIDNIGHT AIR STATE PTE LIMITED,2023,OS/OA/OP/OM Trace,0,1
4,MIDNIGHT AIR STATE PTE LIMITED,2023,Winding Up Trace,0,0
5,SUNSHINE GREENYARN PREFECTURE ENTERPRISE,2023,"Litigation Trace: District Court\n($60,000 - $...",0,1
6,SUNSHINE GREENYARN PREFECTURE ENTERPRISE,2023,"Litigation Trace: High Court\n(> $250,000)",1,0
7,SUNSHINE GREENYARN PREFECTURE ENTERPRISE,2023,"Litigation Trace: Magistrate Court\n(< $60,000)",0,0
8,SUNSHINE GREENYARN PREFECTURE ENTERPRISE,2023,OS/OA/OP/OM Trace,1,1
9,SUNSHINE GREENYARN PREFECTURE ENTERPRISE,2023,Winding Up Trace,0,1


Table 6: Business Lit Search

In [16]:
print('Given Input')
df_dict['Business Lit Search']

Given Input


,SN,Court/Type,Case No.,Year,Filing Date,Plaintiff / Petitioner/Claimant,Case Status,Case Status Date,Case Status Remarks,Plaintiff / Claimant Status,Plaintiff / Claimant Status Date,Defendant Status,Defendant Status Date,Amount(s),Last Updated in Experian Database,Main Cause
0,1,MC/ WOS,106,20XX,12/09/20XX,TRANS \nPTE LTD,PENDING,08/03/20XX,NaN,PENDING,08/03/20XX,PENDING,08/03/20XX,"SGD \n50,000.00",15/08/20XX,CONTRACTS - GENERAL
1,2,DC/ OC,17,20XX,13/04/20XX,SAHA \nCHANDRA,PENDING,20/04/20XX,NaN,PENDING,13/04/20XX,PENDING,13/04/20XX,-,20/04/20XX,CONTRACTS – CREDIT FACILITIES /OVERDRAFT/GURAN...


In [17]:
print('Generated Output')
random_business_lit_search = pd.DataFrame(
    data=random_params['Business Lit Search'], columns=random_params['Business Lit Search'].keys()
)
random_business_lit_search['main_cause_category'] = random_business_lit_search['main_cause'].apply(lambda x: label_litigation_main_cause(litigation_main_cause_clustering_model, str(x)))
random_business_lit_search['case_status_category'] = random_business_lit_search['case_status'].apply(lambda x: label_litigation_case_status(litigation_case_status_clustering_model, str(x)))
random_business_lit_search['plaintiff_status_category'] = random_business_lit_search['plaintiff_status'].apply(lambda x: label_litigation_party_status(litigation_party_status_clustering_model, str(x)))
random_business_lit_search['defendant_status_category'] = random_business_lit_search['defendant_status'].apply(lambda x: label_litigation_party_status(litigation_party_status_clustering_model, str(x)))
random_business_lit_search
# print(random_params['Business Lit Search'])

Generated Output


,sn,court_type,case_no,year,filing_date,plaintiff,case_status,case_status_date,case_status_remarks,plaintiff_status,...,defendant_status,defendant_status_date,amount,last_update,main_cause,individual_id,main_cause_category,case_status_category,plaintiff_status_category,defendant_status_category
0,1,DC / WOS,6902,2007,2007/07/15,SUNSET METAL PREFECTURE PTE LIMITED,WRIT OF SUMMONS,2014/05/07,NaN,UNDISCHARGED BANKRUPT,...,201135549E,2015/05/04,SGD 481393,2017/09/20,Others(CLAIM FOR RETURN OF INVESTMENT MONIES),R607XXXXK,Others,Others,Bankruptcy Related,Others
1,2,DC / WOS,2712,2022,2022/07/13,SUNSHINE AIR COAST ENTERPRISE,STAY OF PROCEEDINGS,2022/11/19,NaN,ISCONTINUED/WITHDRAWN,...,49555135,2022/07/18,SGD 460189,2023/01/02,Contract-Others(Breach of investment agreement...,R607XXXXK,Contracts,Others,Others,Others
2,3,SC / WOS,7333,2009,2009/11/17,SUNSET METAL COAST CORP,CONCLUDED (07-MAY-2012),2017/02/13,NaN,DISCONTINUED/WITHDRAWN,...,NaN,2011/01/21,SGD 199709,2020/10/12,Contract-Others(Breach of Deed of Settlement),R607XXXXK,Contracts,Concluded,Others,Others
3,4,SC / WOS,9341,2004,2004/11/05,MOONLIGHT WATER CENTRAL ENTERPRISE,CONCLUDED (28-MAR-2012),2012/03/20,NaN,SETTLED â€“ BEFORE SET DOWN,...,DISCONTINUED/WITHDRAWN(08-NOV-2013),2014/07/17,SGD 940068,2016/02/22,"Banking, Credit and Security-Others(TERM LOAN)",R607XXXXK,Financials,Concluded,Others,Others
4,5,DC / WOS,3707,2008,2008/02/05,SUNSET WATER STATE CORP,SERVICE OF THE WRIT,2009/03/24,NaN,NaN,...,SETTLED â€“ BEFORE SET DOWN,2012/11/23,SGD 894887,2016/05/09,Others(TORTIOUS ACT),R607XXXXK,Others,Others,Others,Others


Table 7: Bankruptcy Records

In [18]:
print('Given Input')
df_dict['Bankruptcy Records']

Given Input


,Case No.,Year,Filing Date,Plaintiff / Petitioner/ Claimant,Case Status,Case Status Date,Case Status Remarks,Plaintiff / Claimant Status,Plaintiff / Claimant Status Date,Defendant Status,Defendant Status Date,Amount(s),Last Updated in Experian Database,Name of Private Trustees Appointed (As per Experian Database)
0,126,20XX,29/09/20XX,ONE BANK LTD,WITHDRAWN,03/08/20XX,NaN,WITHDRAWN,03/08/20XX,DISCHARGE,03/08/20XX,"SGD 50,000.00",15/01/20XX,TAN EE EE \nABC PARTNER LLC \n5 SHENTON WAY #0...


In [19]:
print('Generated Output')
random_bankruptcy_rec = pd.DataFrame(
    data=random_params['Bankruptcy Records'], columns=random_params['Bankruptcy Records'].keys()
)
random_bankruptcy_rec['case_status_category'] = random_bankruptcy_rec['case_status'].apply(lambda x: label_litigation_case_status(litigation_case_status_clustering_model, str(x)))
random_bankruptcy_rec['plaintiff_status_category'] = random_bankruptcy_rec['plaintiff_status'].apply(lambda x: label_litigation_party_status(litigation_party_status_clustering_model, str(x)))
random_bankruptcy_rec['defendant_status_category'] = random_bankruptcy_rec['defendant_status'].apply(lambda x: label_litigation_party_status(litigation_party_status_clustering_model, str(x)))
random_bankruptcy_rec


Generated Output


,case_no,year,filing_date,plaintiff,case_status,case_status_date,case_status_remarks,plaintiff_status,plaintiff_status_date,defendant_status,defendant_status_date,amount,last_update,name_of_private_trustees,individual_id,case_status_category,plaintiff_status_category,defendant_status_category
0,5420,2014,2014/12/18,DUSK RIVER INTERNATIONAL PTE LIMITED,APPLICATION FOR DISCHARGE,2021/09/25,NaN,FINAL JUDGMENT WITHOUT TRIAL,2016/03/11,FINAL JUDGMENT WITHOUT TRIAIL,2018/07/07,SGD 569402,2021/10/15,SUSAN MEI ABEL PUNGGOL BROADWAY #26-25 SINGAPO...,R607XXXXK,Discharged,Judgement,Judgement
1,251,2018,2018/12/17,SUNSHINE RIVER COAST CORP,NOTICE OF DISCHARGE UNDER SECTION125(3),2020/03/29,NaN,NOTICE OF ANNUMENT UNDER SECTION 123A,2020/09/25,CONCLIDED,2019/11/28,SGD 931301,2020/12/30,TOM LEE PATHERSON CALDECOTT MAISON #23-37 SING...,R607XXXXK,Discharged,Discharge Notice,Others


Table 8: Individual Lit Search

In [20]:
print('Given Input')
df_dict['Individual Lit Search']

Given Input


,SN,Court/ Type,Case No.,Year,Filing Date,Plaintiff/ Petitioner/Claimant,Case Status,Case Status Date,Case Status Remarks,Plaintiff / Claimant Status,Plaintiff / Claimant Status Date,Defendant Status,Defendant Status Date,Amount(s),Last Updated in Experian Database,Main Cause
0,1,SC/ WOS,33,20XX,13/03/20XX,TAN SENG \nHONG,PENDING TRANSFERRED FROM DC 106/20XX,NaN,NaN,PENDING,13/03/20XX,PENDING,13/03/20XX,"1) SGD 57,330.00 \n2) SGD 45,000.00",21/03/20XX,CONTRACTS - GENERAL
1,2,DC/ WOS,106,20XX,12/09/20XX,TAN SENG \nHONG,TRANSFERRED,13/03/20XX,TRANSFERRED TO HC 33/20XX,-,NaN,-,NaN,"1) SGD 50,000.00",20/10/20XX,CONTRACTS - GENERAL
2,3,DC/ OC,106,20XX,13/04/20XX,SAHA \nCHANDRA,PENDING,20/04/20XX,NaN,PENDING,13/04/20XX,PENDING,13/04/20XX,-,22/04/20XX,CONTRACTS - GENERAL


In [21]:
print('Generated Output')
random_indv_lit_search = pd.DataFrame(
    data=random_params['Individual Lit Search'], columns=random_params['Individual Lit Search'].keys()
)
random_indv_lit_search['main_cause_category'] = random_indv_lit_search['main_cause'].apply(lambda x: label_litigation_main_cause(litigation_main_cause_clustering_model, str(x)))
random_indv_lit_search['case_status_category'] = random_indv_lit_search['case_status'].apply(lambda x: label_litigation_case_status(litigation_case_status_clustering_model, str(x)))
random_indv_lit_search['plaintiff_status_category'] = random_indv_lit_search['plaintiff_status'].apply(lambda x: label_litigation_party_status(litigation_party_status_clustering_model, str(x)))
random_indv_lit_search['defendant_status_category'] = random_indv_lit_search['defendant_status'].apply(lambda x: label_litigation_party_status(litigation_party_status_clustering_model, str(x)))
random_indv_lit_search

Generated Output


,sn,type,case_no,year,filing_date,plaintiff,case_status,case_status_date,case_status_remarks,plaintiff_status,...,defendant_status,defendant_status_date,amount,last_update,main_cause,individual_id,main_cause_category,case_status_category,plaintiff_status_category,defendant_status_category
0,1,SC / WOS,8720,2006,2006/05/29,GOTHRY BIN PATHERSON,CONCLUDED (12-DEC-2013),2013/06/25,NaN,NOTICE OF DISCHARGE UNDER SECTION 125(3),...,CONLUDED,2013/02/28,SGD 623752,2016/12/08,Others(Companies),R607XXXXK,Provision of Goods and Services,Concluded,Discharge Notice,Others
1,2,DC / OC,8580,2015,2015/04/01,GOTHRY BIN NAQUILA,SET ASIDE,2017/02/06,NaN,ADJOURNED,...,SUMMARY JUDGMENT -O14,2016/06/12,SGD 727687,2018/12/01,Others(Bills of Exchange Act; Dishonoured Cheque),R607XXXXK,Others,Others,Others,Judgement


Table 9: Accreditation

In [22]:
print('Given Input')
df_dict['Accreditation']

Given Input


,Business,Ranking Source,Ranks
0,ABC HOLDINGS PTE LTD,Singapore1000/SME1000 Rankings,2008 S1000: Ranked 1456
1,WWW PTE LTD,Singapore1000/SME1000 Rankings,2008 S1000: Ranked 200
2,T.A.K. SPECIALIST,Singapore1000/SME1000 Rankings,-


In [23]:
print('Generated Output')
random_accreditation = pd.DataFrame(
    data=random_params['Accreditation'], columns=random_params['Accreditation'].keys()
)
random_accreditation

Generated Output


,business,award_name,award_ranking
0,MOONLIGHT PLASTIC REGIONAL,Enterprise 50 (E50),2012 Ranked 314
1,DUSK AIR PREFECTURE,Enterprise 50 (E50),2006 Ranked 465
2,DAWN WOOD REGIONAL,Singapore 1000,2016 Ranked 523
3,DAWN PLASTIC REGIONAL,Singapore Business Awards,2019 Ranked 952


Table 10: Financial Related Search Count  

In [24]:
print('Given Input')
df_dict['Financial Related Search Count']

Given Input


,Year,Total,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,20XX,3.0,3,0,0,0,0,0,0,0,0,0,0,0
1,Trade Insurance,NaN,1,0,0,0,0,0,0,0,0,0,0,0
2,Financial Institution,NaN,2,0,0,0,0,0,0,0,0,0,0,0
3,20XX,3.0,0,0,0,0,0,1,0,1,0,0,0,1
4,20XX,0.0,0,0,0,0,0,0,0,0,0,0,0,0


In [25]:
print('Generated Output')
random_fin_rel_search_count = random_Related_Search_Count_Generation(ENTITY_ID, random_params['Financial Related Search Count']['purpose'], req_date.year-1)
random_fin_rel_search_count

Generated Output


,individual_id,year,month,purpose,count
0,R607XXXXK,2022,All,All,4
1,R607XXXXK,2022,Jan,All,0
2,R607XXXXK,2022,Feb,All,0
3,R607XXXXK,2022,Mar,All,0
4,R607XXXXK,2022,Apr,All,0
5,R607XXXXK,2022,May,All,3
6,R607XXXXK,2022,Jun,All,0
7,R607XXXXK,2022,Jul,All,0
8,R607XXXXK,2022,Aug,All,0
9,R607XXXXK,2022,Sep,All,0


Table 11: Commercial Related Search Count

In [26]:
print('Given Input')
df_dict['Commercial Related Search Count']

Given Input


,Year,Total,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,20XX,1.0,1,0,0,0,0,0,0,0,0,0,0,0
1,Commerce-Retail,NaN,1,0,0,0,0,0,0,0,0,0,0,0
2,20XX,3.0,0,0,0,0,0,1,0,1,0,0,0,1
3,20XX,0.0,0,0,0,0,0,0,0,0,0,0,0,0


In [27]:
print('Generated Output')
random_comm_rel_search_count = random_Related_Search_Count_Generation(ENTITY_ID, random_params['Commercial Related Search Count']['purpose'], req_date.year-1)
random_comm_rel_search_count

Generated Output


,individual_id,year,month,purpose,count
0,R607XXXXK,2022,All,All,4
1,R607XXXXK,2022,Jan,All,0
2,R607XXXXK,2022,Feb,All,3
3,R607XXXXK,2022,Mar,All,1
4,R607XXXXK,2022,Apr,All,0
...,...,...,...,...,...
60,R607XXXXK,2020,Aug,All,0
61,R607XXXXK,2020,Sep,All,0
62,R607XXXXK,2020,Oct,All,0
63,R607XXXXK,2020,Nov,All,0


Table 12: Legal Related Search Count 

In [28]:
print('Given Input')
df_dict['Legal Related Search Count']

Given Input


,Year,Total,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,20XX,2,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Conveyancing,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,Others,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20XX,3,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
4,20XX,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
print('Generated Output')
random_leg_rel_search_count = random_Related_Search_Count_Generation(ENTITY_ID, random_params['Legal Related Search Count']['purpose'], req_date.year-1)
random_leg_rel_search_count

Generated Output


,individual_id,year,month,purpose,count
0,R607XXXXK,2022,All,All,4
1,R607XXXXK,2022,Jan,All,0
2,R607XXXXK,2022,Feb,All,1
3,R607XXXXK,2022,Mar,All,1
4,R607XXXXK,2022,Apr,All,0
5,R607XXXXK,2022,May,All,0
6,R607XXXXK,2022,Jun,All,0
7,R607XXXXK,2022,Jul,All,2
8,R607XXXXK,2022,Aug,All,0
9,R607XXXXK,2022,Sep,All,0


Export the randomly generated data

In [30]:
# Output Data
writer = pd.ExcelWriter(f'../../Outputs/IBS_Random_Output/{ENTITY_ID}.xlsx', engine='xlsxwriter')

random_req_criteria.to_excel(writer, sheet_name='Request Criteria')
random_search_rec.to_excel(writer, sheet_name='Search Record')
random_current_business.to_excel(writer, sheet_name='Current Business')
randon_previous_business.to_excel(writer, sheet_name='Previous Business')
randomm_curr_business_lit_trace.to_excel(writer, sheet_name='Current Business Lit Trace')
random_business_lit_search.to_excel(writer, sheet_name='Business Lit Search')
random_bankruptcy_rec.to_excel(writer, sheet_name='Bankruptcy Record')
random_indv_lit_search.to_excel(writer, sheet_name='Individual Lit Search')
random_accreditation.to_excel(writer, sheet_name='Accreditation')
random_fin_rel_search_count.to_excel(writer, sheet_name='Financial Related Search Count')
random_comm_rel_search_count.to_excel(writer, sheet_name='Commercial Related Search Count')
random_leg_rel_search_count.to_excel(writer, sheet_name='Legal Related Search Count')

writer.close()